# Exercise 12.3: LLM Temperature Token Sampling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/Maths-for-IT-Notebooks/blob/main/Chapter-12/03-temperature-token-sampling.ipynb)

**Book Reference:** Chapter 12 — Conditional Probability and Random Variables (`chapters/13-conditional-probability-and-random-variables.tex`, Section 4)  
**Estimated Time:** 15–20 min

### Learning Objectives
- Convert raw LLM logits into next-token conditional probabilities using temperature-scaled softmax.
- Sample next-token IDs from the resulting categorical distribution.
- Measure how increasing temperature $T$ increases distribution spread (variance) and lowers peak probability.


In [ ]:
import numpy as np

### Mathematical Context

At inference time, a large language model outputs raw scores (logits) $\mathbf{z} = (z_1, \dots, z_V)$ conditioned on the prompt $w_1 \dots w_{t-1}$. Temperature $T > 0$ scales the logits before applying softmax:
$$P_T(w_t = i \mid \text{prompt}) = \frac{\exp((z_i - m) / T)}{\sum_{j=1}^V \exp((z_j - m) / T)}, \quad m = \max_j z_j.$$

- **Low temperature** ($T < 1$): sharpens the distribution around the top token (low sampling variance).
- **High temperature** ($T > 1$): flattens the distribution toward uniform (higher sampling variance, more diverse/creative outputs).

In [ ]:
def softmax_with_temperature(logits: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    """Compute numerically stable softmax probabilities scaled by temperature T.

    Formula:
        scaled = (logits - max(logits)) / temperature
        probs = exp(scaled) / sum(exp(scaled))

    Args:
        logits: 1D NumPy array of unnormalized log-scores.
        temperature: Positive temperature parameter T > 0.

    Returns:
        1D NumPy probability array summing to 1.0.
    """
    # TODO: Validate temperature > 0, shift logits by max(logits), divide by temperature,
    # exponentiate, and normalize by the sum.
    raise NotImplementedError("Complete this TODO")


def sample_tokens(
    logits: np.ndarray, temperature: float = 1.0, num_samples: int = 1000, seed: int = 42
) -> np.ndarray:
    """Sample token indices from the temperature-scaled next-token distribution.

    Args:
        logits: 1D NumPy array of logits over vocabulary {0, ..., V-1}.
        temperature: Positive temperature parameter T > 0.
        num_samples: Number of independent draws.
        seed: Random seed (default 42).

    Returns:
        1D integer NumPy array of sampled token indices of length num_samples.
    """
    # TODO: Compute probs via softmax_with_temperature and draw indices via rng.choice(len(probs), size=num_samples, p=probs).
    raise NotImplementedError("Complete this TODO")


def token_distribution_spread(probs: np.ndarray) -> tuple[float, float]:
    """Summarize concentration and spread of a categorical distribution.

    Args:
        probs: 1D probability array over token indices 0..V-1.

    Returns:
        Tuple (max_prob, index_variance) where max_prob = max(probs) and
        index_variance = sum((i - E[i])^2 * probs[i]).
    """
    # TODO: Compute max_prob and variance of token index i = 0..V-1 under probs.
    raise NotImplementedError("Complete this TODO")

<details>
<summary>Hint 1: Numerically stable softmax</summary>

Subtract `np.max(z)` before dividing by `temperature` and calling `np.exp(...)`. That prevents overflow while leaving the normalized probabilities mathematically unchanged.

</details>

<details>
<summary>Hint 2: Sampling and spread</summary>

Use `rng.choice(len(probs), size=num_samples, p=probs)` for sampling. For spread, treat token indices `np.arange(len(probs))` as the random variable values.

</details>

In [ ]:
# Self-check assertions
vocab = ["cat", "dog", "bird", "fish"]
logits = np.array([3.0, 2.0, 1.0, 0.0])

p_low = softmax_with_temperature(logits, temperature=0.2)
p_mid = softmax_with_temperature(logits, temperature=1.0)
p_high = softmax_with_temperature(logits, temperature=2.5)

assert abs(float(np.sum(p_mid)) - 1.0) < 1e-12

max_low, var_low = token_distribution_spread(p_low)
max_mid, var_mid = token_distribution_spread(p_mid)
max_high, var_high = token_distribution_spread(p_high)

assert max_low > max_mid > max_high, "Peak probability should decrease as T increases"
assert var_low < var_mid < var_high, "Sampling variance should increase as T increases"

draws_low = sample_tokens(logits, temperature=0.2, num_samples=1000, seed=42)
draws_high = sample_tokens(logits, temperature=2.5, num_samples=1000, seed=42)
assert np.mean(draws_low == 0) > 0.95
assert np.mean(draws_high == 0) < 0.60

for T, p, (mx, vr) in [(0.2, p_low, (max_low, var_low)), (1.0, p_mid, (max_mid, var_mid)), (2.5, p_high, (max_high, var_high))]:
    print(f"T={T:3.1f} | probs={np.round(p, 3)} | max_p={mx:.3f} | var={vr:.3f}")
print("All assertions passed!")

### Think Deeper

1. What happens to `softmax_with_temperature(logits, temperature)` in the limits $T \to 0^+$ and $T \to \infty$?
2. Why might a coding assistant use low temperature ($T \approx 0.1$–$0.2$) for unit-test generation, while a creative writing assistant uses higher temperature ($T \approx 0.8$–$1.0$)?